# Notebook 안내
원본 팀 연구 코드의 계산 로직은 변경하지 않았다. 가독성을 위한 설명 셀만 추가했으며 개인정보·데이터 노출 방지를 위해 기존 실행 출력은 제거했다. 원천 데이터는 포함하지 않으며 독립 재실행은 하지 않았다. README의 결과 및 한계 참고.


# 총주주환원 팩터 롱숏 전략배당수익률과 순주식발행을 합산한 총주주환원수익률(SY)로 롱을, 순주식발행(NSI)으로 숏을 구성하는 롱숏 전략의 백테스트다.측정 구간은 2015년 7월부터 2026년 5월까지 130개월이며, 모든 수치는 거래비용과 대차비용 차감 후다.## 필요한 데이터 파일아래 여덟 개 CSV를 노트북과 같은 위치의 `Data/` 폴더에 둔다. 모두 DataGuide에서 받은 열=종목명 와이드 형식이다.| 파일 | 내용 | 비고 || --- | --- | --- || `KOSPIKOSDAQ_수정주가.csv` | 수정주가 (배당 미반영) | 배당수익률 분모, 주식수 역산 || `수정주가_현금배당.csv` | 수정주가 (현금배당 반영) | 수익률 계산용 || `시가총액_금융포함.csv` | 시가총액 (백만원) | 금융업 포함, 2015-04부터 || `시가총액.csv` | 시가총액 (백만원) | 금융업 미포함, 과거 구간 보완용 || `KOSPIKOSDAQ_DPS.csv` | 주당배당금 | 액면분할 조정된 값 || `거래대금_금융포함.csv` | 월말 1일 거래대금 (원) | 금융업 포함, 월별 || `KOSPI200_YN.csv` | KOSPI200 구성 여부 | 월말 Y/N || `factor_model_index.csv` | KOSPI·SMB·HML 지수, Rf | Rf는 연율 퍼센트 |## 실행 순서셀을 위에서 아래로 차례대로 실행하면 된다. 2번 셀의 `CFG` 딕셔너리만 바꾸면 모든 파라미터를 조정할 수 있다.

## 1. 설정모든 파라미터가 여기 모여 있다. 값을 바꾸고 아래 셀들을 다시 실행하면 해당 구성의 성과가 나온다.

### 코드 설명 — 환경 및 설정: 전략 기간, 비용, lag와 유니버스 조건을 정의한다.


In [ ]:
import warnings, refrom pathlib import Pathimport numpy as npimport pandas as pdfrom scipy import statsimport matplotlib.pyplot as pltimport matplotlib as mplwarnings.filterwarnings("ignore")pd.set_option("display.width", 160)pd.set_option("display.max_columns", 40)# 한글 폰트 (없으면 무시하고 진행)for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:    try:        mpl.font_manager.findfont(_f, fallback_to_default=False)        mpl.rcParams["font.family"] = _f        break    except Exception:        passmpl.rcParams["axes.unicode_minus"] = FalseDATA = Path("Data")CFG = dict(    # 기간    START = "2015-07-31",    END   = "2026-05-31",    # 유니버스    UNIV_Q      = 0.70,    # 시가총액 상위 30% (하위 70% 분위수 이상)    ADV_WINDOW  = 6,       # 거래대금 평균 창 (개월)    ADV_CUT     = 0.10,    # 거래대금 하위 10% 배제    USE_ADV     = True,    # 거래대금 필터 사용 여부    # 롱 건전성 필터: 최근 회계연도 배당 지급 + 3년 비감소    USE_PAYOUT_FILTER = True,    NOCUT_TOL   = 0.999,   # 비감소 판정 허용오차 (0.1%)    USE_NODIL   = False,   # 3년 연속 비희석 조건 추가 여부    # 포트폴리오    N_LONG      = 20,    N_SHORT     = 20,    SHORT_W     = 0.80,    # 숏 명목비중 (롱 = 1.00)    SCHEME      = "iv",    # "iv" 역변동성 | "eq" 동일가중 | "mc" 시가총액가중    VOL_WINDOW  = 24,      # 역변동성 창 (개월)    WEIGHT_CAP  = None,    # 개별 비중 상한 (예: 0.10), None이면 제한 없음    FIN_CAP     = 6,       # 금융업 한쪽당 최대 종목 수    FREQ        = "Q",     # "Q" 분기(3·6·9·12월 말) | "M" 월간    # 비용    RT_BP       = 50,      # 왕복 거래비용 (bp)    BORROW_ANN  = 0.03,    # 대차비용 연율    # 공매도 금지 처리: "A" 롱온리 | "B" 금지 무시 | "C" 지수선물 헤지    BAN_MODE    = "A",    BAN_PERIODS = [("2020-03-01", "2021-04-30"), ("2023-11-01", "2025-03-31")],    # 신호    DPS_LAG     = 15,      # DPS 회계 스탬프 보정 (개월)    NSI_CLIP    = 0.30,    # |NSI| 상한, 초과 시 결측 (인적분할·감자 제외)    LONG_SIGNAL = "SY",    # "SY" 배당+소각 | "DY" 배당 단독 | "NSI" 소각 단독)FIN_PATTERN = r"(금융지주|은행|뱅크|증권|보험|화재|해상|캐피탈|카드|저축|자산운용|신탁|코리안리|생명(?!과학|공학|자원|정보))"

## 2. 데이터 로드모든 계열을 `resample("ME").last()`로 월말 기준에 맞추고 종목명 교집합을 취한다.시가총액은 두 파일을 이어 붙인다. 기존 `시가총액.csv`는 3,535열로 금융업이 통째로 빠져 있어 그대로 쓰면은행지주·보험·증권이 유니버스에서 조용히 제외된다. 금융 포함 파일을 우선하고 겹치지 않는 과거 구간만기존 파일로 채운다. 두 파일은 겹치는 구간의 값이 완전히 일치한다.

### 코드 설명 — DataGuide 패널 병합: 시총 금융업 포함 파일 우선, 기존 파일 결측 보완, 월말 및 종목 열 정렬. 입력 파일명 불일치 확인 필요.


In [ ]:
def load(name, **kw):    return pd.read_csv(DATA / name, index_col=0, parse_dates=True, **kw).resample("ME").last()pr_raw  = load("KOSPIKOSDAQ_수정주가.csv")          # 배당 미반영tr_raw  = load("수정주가_현금배당.csv")              # 배당 반영mc_fin  = load("시가총액_금융포함.csv")mc_old  = load("시가총액.csv")dps_raw = load("KOSPIKOSDAQ_DPS.csv")k2_raw  = load("KOSPI200_YN.csv").eq("Y")fi      = load("factor_model_index.csv")adv_raw = pd.read_csv(DATA / "거래대금_금융포함.csv", index_col=0, parse_dates=True)adv_raw = adv_raw.loc[:, ~adv_raw.columns.duplicated()]TICKERS = sorted(set(pr_raw.columns) & set(tr_raw.columns) & set(dps_raw.columns)                 & set(k2_raw.columns) & set(mc_fin.columns))IDXM = pr_raw.indexpr  = pr_raw[TICKERS]tr  = tr_raw[TICKERS].reindex(IDXM)dps = dps_raw[TICKERS].reindex(IDXM)k2  = k2_raw[TICKERS].reindex(IDXM).fillna(False)mc  = (mc_fin.reindex(index=IDXM, columns=TICKERS)       .combine_first(mc_old.reindex(index=IDXM, columns=TICKERS)))adv_me = adv_raw.reindex(index=IDXM, columns=TICKERS)ret = tr.pct_change()                                  # 월별 총수익률fi  = fi.reindex(IDXM).ffill(limit=1)rf  = fi["Rf"] / 100 / 12mkt = fi["KOSPI"].pct_change()smb = fi["SMB"].pct_change()hml = fi["HML"].pct_change()FINS = {c for c in TICKERS if re.search(FIN_PATTERN, c)}print(f"종목 수 {len(TICKERS):,} · 기간 {IDXM.min().date()} ~ {IDXM.max().date()}")print(f"금융업으로 식별된 종목 {len(FINS)}개")_valid = mc.loc["2022-06-30"].notna()print(f"거래대금 커버리지 (2022-06, 시가총액 유효 종목 기준) "      f"{adv_me.loc['2022-06-30'][_valid].notna().mean():.1%}")

## 3. 팩터 계산```DY(i,t)  = DPS(i, FY) / 수정주가(i,t)          # DPS는 15개월 shiftNSI(i,t) = ln( 수정주식수(i,t-1) / 수정주식수(i,t-13) )SY(i,t)  = DY(i,t) + ( -NSI(i,t) )```상장주식수 원계열을 그대로 쓰면 액면분할과 무상증자가 희석으로 잡힌다. 수정주가가 이미 분할과 무상증자를반영하고 있으므로 `수정주식수 = 시가총액 / 수정주가`로 역산한다. 삼성전자의 2018년 5월 50대 1 분할이이 계열에서는 사라지고 같은 해 자사주 소각분만 남는지로 검증한다.DPS는 회계연도 Y의 값이 Y년 1월부터 기재되지만 실제 공시는 Y+1년 3월 말이므로 15개월 shift한다.순주식발행은 시장 데이터라 회계 lag가 필요 없고 1개월 shift만 적용한다.

### 코드 설명 — 기업행동 및 배당 정보: 시총/수정주가로 수정주식수 대용치를 생성하고 DPS lag 및 NSI를 계산한다.


In [ ]:
D   = dps.shift(CFG["DPS_LAG"])                        # 사용 가능한 회계연도 DPSDY  = (D / pr).where(pr > 0)shares = (mc * 1e6) / pr                               # 수정주식수 역산NSI = (np.log(shares / shares.shift(12))       .replace([np.inf, -np.inf], np.nan)       .where(lambda x: x.abs() <= CFG["NSI_CLIP"])       .shift(1))SY   = DY.fillna(0) + (-NSI).fillna(0)VOL  = ret.rolling(CFG["VOL_WINDOW"]).std()# 검증: 삼성전자 수정주식수 (분할이 제거되고 소각만 남아야 함)chk = shares["삼성전자"].loc[["2017-06-30", "2018-06-30", "2018-12-31"]] / 1e8print("삼성전자 수정주식수 (억주)")for d, v in chk.items():    print(f"  {d.date()}  {v:,.2f}")print(f"  2018년 6월 대비 12월 변화 {chk.iloc[2]/chk.iloc[1]-1:+.1%}  (자사주 소각분)")

## 4. 유니버스와 롱·숏 풀유니버스는 매월 말 재산정하는 시가총액 상위 30퍼센트다. 그 위에 롱과 숏에 각각 다른 제약을 얹는다.롱 제약은 배당의 지속성이다. 넓은 유니버스에서 배당수익률이 높게 나오는 경로는 둘인데, 하나는 회사가 실제로많이 주는 것이고 다른 하나는 분모인 주가가 빠진 것이다. 최근 회계연도에 배당을 지급했고 최근 세 개 회계연도의주당배당금이 감소하지 않았을 것을 요구하면 후자가 걸러진다.숏 제약은 규제에서 나온다. 2021년 5월 3일부터 2023년 11월 5일까지 한국은 KOSPI200과 KOSDAQ150구성종목에 한해서만 공매도를 허용했다. 숏 풀을 KOSPI200으로 두면 이 문제가 정의상 발생하지 않는다.거래대금 필터는 실행 가능성 요건이다. 거래대금 파일이 월말 하루치 값이므로 6개월 평균을 ADV 프록시로 쓴다.

### 코드 설명 — 시점별 유니버스 및 후보 필터: 시총, 거래대금, 배당 지속성, KOSPI200 구성 여부를 적용한다.


In [ ]:
ALIVE = pr.notna() & mc.notna() & (mc > 0) & ret.notna()UNIV  = ALIVE & mc.ge(mc.where(ALIVE).quantile(CFG["UNIV_Q"], axis=1), axis=0)# 거래대금 필터 (임계값은 유니버스 횡단면에서 산출, 결측은 통과)ADV = adv_me.rolling(CFG["ADV_WINDOW"], min_periods=max(2, CFG["ADV_WINDOW"] // 2)).mean()if CFG["USE_ADV"]:    thr = ADV.where(UNIV).quantile(CFG["ADV_CUT"], axis=1)    LIQ = ADV.ge(thr, axis=0) | ADV.isna()else:    LIQ = pd.DataFrame(True, index=IDXM, columns=TICKERS)# 롱 건전성 필터NOCUT3 = ((D > 0)          & (D >= D.shift(12) * CFG["NOCUT_TOL"])          & (D.shift(12) >= D.shift(24) * CFG["NOCUT_TOL"]))NODIL3 = (NSI <= 0.01) & (NSI.shift(12) <= 0.01) & (NSI.shift(24) <= 0.01)LONG_OK = pd.DataFrame(True, index=IDXM, columns=TICKERS)if CFG["USE_PAYOUT_FILTER"]:    LONG_OK &= NOCUT3.fillna(False)if CFG["USE_NODIL"]:    LONG_OK &= NODIL3.fillna(False)POOL_L = UNIV & LIQ & LONG_OK & DY.notna() & NSI.notna()POOL_S = ALIVE & k2 & LIQ & NSI.notna()_sl = slice(CFG["START"], CFG["END"])print(f"유니버스        평균 {UNIV.loc[_sl].sum(1).mean():6.0f}종목  "      f"({UNIV.loc[_sl].sum(1).min():.0f} ~ {UNIV.loc[_sl].sum(1).max():.0f})")print(f"롱 풀           평균 {POOL_L.loc[_sl].sum(1).mean():6.0f}종목")print(f"숏 풀 (KOSPI200) 평균 {POOL_S.loc[_sl].sum(1).mean():6.0f}종목")if CFG["USE_ADV"]:    _t = ADV.where(UNIV).quantile(CFG["ADV_CUT"], axis=1).loc[_sl]    print(f"거래대금 임계값   중앙값 {_t.median()/1e8:.1f}억원/일  ({_t.min()/1e8:.1f} ~ {_t.max()/1e8:.1f}억)")

## 5. 성과지표와 회귀`statsmodels` 없이도 돌아가도록 Newey-West HAC 표준오차를 직접 구현했다. maxlags는 12개월이다.

### 코드 설명 — 통계 검증: NumPy로 OLS 및 Newey–West HAC(12)를 구현하고 SciPy로 양측 p값을 계산한다.


In [ ]:
def performance_metrics(r, rf_s=None, freq=12):    r = r.dropna()    n = len(r)    cum  = (1 + r).prod()    cagr = cum ** (freq / n) - 1    vol  = r.std(ddof=1) * np.sqrt(freq)    ex   = r if rf_s is None else (r - rf_s.reindex(r.index))    sharpe = ex.mean() / ex.std(ddof=1) * np.sqrt(freq)    eq  = (1 + r).cumprod()    dd  = eq / eq.cummax() - 1    return dict(CAGR=cagr, Vol=vol, Sharpe=sharpe, MDD=dd.min(),                MDD_date=dd.idxmin(), NegMonths=(r < 0).mean(),                Worst=r.min(), Best=r.max(), N=n)def nw_ols(y, X, maxlags=12):    """Newey-West HAC 표준오차 OLS. X에 상수항은 포함하지 않는다."""    y = np.asarray(y, float)    X = np.column_stack([np.ones(len(y)), np.asarray(X, float)])    n, k = X.shape    XtX_inv = np.linalg.inv(X.T @ X)    b = XtX_inv @ (X.T @ y)    e = y - X @ b    Xe = X * e[:, None]    S = Xe.T @ Xe    for l in range(1, maxlags + 1):        w = 1 - l / (maxlags + 1)        G = Xe[l:].T @ Xe[:-l]        S += w * (G + G.T)    V = XtX_inv @ S @ XtX_inv * n / (n - k)    se = np.sqrt(np.diag(V))    t = b / se    p = 2 * (1 - stats.t.cdf(np.abs(t), n - k))    ss_res = (e ** 2).sum()    ss_tot = ((y - y.mean()) ** 2).sum()    return dict(coef=b, se=se, t=t, p=p, r2=1 - ss_res / ss_tot, n=n)DATES  = ret.loc[CFG["START"]:CFG["END"]].indexRDATES = DATES[1:]                                     # 수익률이 실현되는 달_MKT = mkt.reindex(RDATES) - rf.reindex(RDATES)_SMB = smb.reindex(RDATES)_HML = hml.reindex(RDATES)REG_OK = _MKT.notna() & _SMB.notna() & _HML.notna()X_FF3  = np.column_stack([_MKT[REG_OK], _SMB[REG_OK], _HML[REG_OK]])X_CAPM = np.column_stack([_MKT[REG_OK]])BANNED = pd.Series(False, index=IDXM)for a, b_ in CFG["BAN_PERIODS"]:    BANNED.loc[a:b_] = Trueprint(f"백테스트 {RDATES.min().date()} ~ {RDATES.max().date()} · {len(RDATES)}개월")print(f"공매도 금지 {int(BANNED.reindex(DATES[:-1]).sum())}개월")

## 6. 백테스트 엔진리밸런싱일에 신호 상위 N종목을 뽑고 가중치를 부여한 뒤, 다음 달 수익률을 적용한다.리밸런싱 사이에는 비중을 드리프트시키되 총노출을 유지하도록 정규화한다.비용은 두 가지다. 거래비용은 매수와 매도 명목금액의 합에 편도 25bp를 곱해 차감하고,대차비용은 숏 명목금액에 월 0.25퍼센트를 부과한다.

### 코드 설명 — 백테스트: 보유기간, 비중, 비용 및 규제 조건을 적용한다.


In [ ]:
def make_weights(names, d, scheme=None, cap=None):    scheme = scheme or CFG["SCHEME"]    cap = CFG["WEIGHT_CAP"] if cap is None else cap    if not names:        return pd.Series(dtype=float)    if scheme == "iv":        v = VOL.loc[d, names].replace(0, np.nan)        w = (1 / v).fillna((1 / v).median())    elif scheme == "mc":        w = mc.loc[d, names].astype(float)        w = w.fillna(w.median())    else:        w = pd.Series(1.0, index=names)    w = w / w.sum()    if cap is not None:                                # 개별 비중 상한        for _ in range(100):            over = w > cap            if not over.any():                break            excess = (w[over] - cap).sum()            w[over] = cap            room = ~over            if not room.any():                break            w[room] += excess * w[room] / w[room].sum()        w = w / w.sum()    return wdef select(signal, d, n, fin_cap):    """신호 상위 n종목. 금융업은 fin_cap개까지만 채택."""    s = signal.loc[d].dropna().nlargest(n * 10)    out, nfin = [], 0    for c in s.index:        if fin_cap is not None and c in FINS:            if nfin >= fin_cap:                continue            nfin += 1        out.append(c)        if len(out) == n:            break    return outdef backtest(long_signal=None, pool_l=None, pool_s=None, **over):    p = {**CFG, **over}    long_signal = long_signal or p["LONG_SIGNAL"]    pool_l = POOL_L if pool_l is None else pool_l    pool_s = POOL_S if pool_s is None else pool_s    raw = {"SY": SY, "DY": DY, "NSI": -NSI}[long_signal]    sig_l = raw.where(pool_l)    sig_s = NSI.where(pool_s)    cost = p["RT_BP"] / 2 / 1e4    borrow = p["BORROW_ANN"] / 12    w = pd.Series(0.0, index=TICKERS)    rows, turns, holdings = [], [], {}    for i, d in enumerate(DATES[:-1]):        nxt = DATES[i + 1]        form = (i == 0) or (True if p["FREQ"] == "M" else d.month in (3, 6, 9, 12))        fut_w = 0.0        if form:            L = select(sig_l, d, p["N_LONG"], p["FIN_CAP"])            tgt = pd.Series(0.0, index=TICKERS)            wl = make_weights(L, d, p["SCHEME"], p["WEIGHT_CAP"])            tgt[wl.index] = wl.values            S = []            if (p["BAN_MODE"] == "B") or (not BANNED[d]):                S = select(sig_s, d, p["N_SHORT"], p["FIN_CAP"])                ws = make_weights(S, d, p["SCHEME"], p["WEIGHT_CAP"]) * p["SHORT_W"]                tgt[ws.index] = tgt[ws.index].values - ws.values            holdings[d] = (L, S)        else:            tgt = w        if p["BAN_MODE"] == "C" and BANNED[d]:            fut_w = p["SHORT_W"]        traded = (tgt - w).abs().sum()        turns.append(traded / 2)        r_ = ((tgt * ret.loc[nxt]).sum(skipna=True)              - traded * cost              - tgt.clip(upper=0).abs().sum() * borrow)        if fut_w > 0:                                   # 지수선물 매도 대체            r_ += (-fut_w * mkt.reindex(IDXM)[nxt]                   - fut_w * (0.01 / 12)                   - (fut_w * 5 / 1e4 if form else 0.0))        rows.append(r_)        drift = tgt * (1 + ret.loc[nxt].fillna(0))        gross = abs(drift.abs().sum())        w = drift if gross < 1e-12 else drift / gross * tgt.abs().sum()    r = pd.Series(rows, index=RDATES)    m = performance_metrics(r, rf)    ex = (r - rf.reindex(RDATES))[REG_OK]    return dict(r=r, metrics=m, ff3=nw_ols(ex, X_FF3), capm=nw_ols(ex, X_CAPM),                turnover=sum(turns) / (len(DATES) / 12) * 100, holdings=holdings)def summarize(res, name=""):    m, f, c = res["metrics"], res["ff3"], res["capm"]    print(f"{name}")    print(f"  CAGR {m['CAGR']:7.2%}   변동성 {m['Vol']:6.2%}   샤프 {m['Sharpe']:.3f}   "          f"MDD {m['MDD']:7.2%} ({m['MDD_date'].date()})")    print(f"  회전율 {res['turnover']:.0f}%   연간 거래비용 {res['turnover']*CFG['RT_BP']/100:.0f}bp   "          f"월간 음수 {m['NegMonths']:.0%}   최악의 월 {m['Worst']:.1%}")    print(f"  CAPM  α {c['coef'][0]:+.2%}/월 (t={c['t'][0]:.2f})   β {c['coef'][1]:+.3f}   R² {c['r2']:.3f}")    print(f"  FF3   α {f['coef'][0]:+.2%}/월 (t={f['t'][0]:.2f})   "          f"MKT {f['coef'][1]:+.3f}  SMB {f['coef'][2]:+.3f}  HML {f['coef'][3]:+.3f}   R² {f['r2']:.3f}")

## 7. 기준안 실행유니버스 시총 상위 30퍼센트, 롱 풀은 배당 지급 및 3년 비감소, 숏 풀은 KOSPI200, 양쪽에 거래대금 하위 10퍼센트 배제,롱 SY 상위 20종목 100퍼센트 / 숏 NSI 상위 20종목 80퍼센트, 역변동성 24개월 가중, 분기 리밸런싱,금융업 한쪽당 6종목, 왕복 50bp, 대차 연 3퍼센트, 공매도 금지월은 롱온리다.

### 코드 설명 — 기준안과 달러중립 비교.


In [ ]:
base = backtest()summarize(base, "기준안 (롱100 / 숏80)")print()neutral = backtest(SHORT_W=1.0)summarize(neutral, "달러중립안 (롱100 / 숏100)")print()kospi = mkt.reindex(RDATES).dropna()mk = performance_metrics(kospi, rf)print(f"KOSPI (배당 미포함)")print(f"  CAGR {mk['CAGR']:7.2%}   변동성 {mk['Vol']:6.2%}   샤프 {mk['Sharpe']:.3f}   MDD {mk['MDD']:7.2%}")

## 8. 국면별·연도별 성과

### 코드 설명 — 기간별·연도별 성과 분석.


In [ ]:
REGIMES = [("2015-07 ~ 2019-12", "2015-07", "2019-12"),           ("2020-01 ~ 2023-12", "2020-01", "2023-12"),           ("2024-01 ~ 2026-05", "2024-01", "2026-05")]rows = []for lab, a, b_ in REGIMES:    p1 = performance_metrics(base["r"].loc[a:b_], rf)    p2 = performance_metrics(neutral["r"].loc[a:b_], rf)    pk = performance_metrics(kospi.loc[a:b_], rf)    rows.append([lab, f"{p1['CAGR']:.2%}", f"{p1['Sharpe']:.3f}", f"{p1['MDD']:.2%}",                 f"{p2['CAGR']:.2%}", f"{p2['Sharpe']:.3f}", f"{pk['CAGR']:.2%}"])print(pd.DataFrame(rows, columns=["구간", "기준안 CAGR", "샤프", "MDD",                                  "중립안 CAGR", "샤프", "KOSPI"]).to_string(index=False))print()ya = base["r"].groupby(base["r"].index.year).apply(lambda x: (1 + x).prod() - 1)yk = kospi.groupby(kospi.index.year).apply(lambda x: (1 + x).prod() - 1)annual = pd.DataFrame({"전략": ya, "KOSPI": yk})print(annual.to_string(float_format=lambda v: f"{v:7.1%}"))

### 코드 설명 — NAV 및 낙폭 시각화.


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True,                       gridspec_kw={"height_ratios": [2, 1]})eq_s = (1 + base["r"]).cumprod()eq_n = (1 + neutral["r"]).cumprod()eq_k = (1 + kospi).cumprod()ax[0].plot(eq_s.index, eq_s, lw=1.6, label="기준안 (롱100/숏80)")ax[0].plot(eq_n.index, eq_n, lw=1.1, label="달러중립안 (롱100/숏100)")ax[0].plot(eq_k.index, eq_k, lw=1.1, label="KOSPI")for a, b_ in CFG["BAN_PERIODS"]:    ax[0].axvspan(pd.Timestamp(a), pd.Timestamp(b_), color="0.85", zorder=0)ax[0].set_yscale("log"); ax[0].set_ylabel("누적 수익 (로그)")ax[0].legend(loc="upper left", frameon=False); ax[0].grid(alpha=.3)ax[0].set_title("누적 성과 (회색 구간은 공매도 금지)")dd = eq_s / eq_s.cummax() - 1ax[1].fill_between(dd.index, dd, 0, color="#c0392b", alpha=.35)ax[1].set_ylabel("낙폭"); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()

## 9. 강건성 검증보유 종목 수, 숏 비중, 섹터 상한, 거래비용, 가중 방식, 리밸런싱 주기를 하나씩 바꿔 돌린다.

### 코드 설명 — 강건성 검증: 파라미터 변경 실험.


In [ ]:
def grid(variants):    rows = []    for label, kw in variants:        res = backtest(**kw)        m, f = res["metrics"], res["ff3"]        rows.append([label, f"{m['CAGR']:.2%}", f"{m['Vol']:.2%}", f"{m['Sharpe']:.3f}",                     f"{m['MDD']:.2%}", f"{f['coef'][0]:+.2%}", f"{f['t'][0]:.2f}",                     f"{res['turnover']:.0f}%"])    return pd.DataFrame(rows, columns=["변형", "CAGR", "변동성", "샤프", "MDD",                                       "FF3 α", "t", "회전율"])print("[롱 종목 수]")print(grid([(f"롱 {n} / 숏 20", dict(N_LONG=n)) for n in (15, 20, 25, 30, 40)]).to_string(index=False))print("\n[숏 종목 수]")print(grid([(f"롱 20 / 숏 {n}", dict(N_SHORT=n)) for n in (10, 15, 20, 30, 50)]).to_string(index=False))print("\n[숏 비중]")print(grid([(f"숏 {int(s*100)}%", dict(SHORT_W=s)) for s in (0.6, 0.7, 0.8, 0.9, 1.0)]).to_string(index=False))

### 코드 설명 — 섹터·가중·비용 등의 대안 검토.


In [ ]:
print("[금융 섹터 상한]")print(grid([(f"상한 {c if c else '없음'}", dict(FIN_CAP=c)) for c in (4, 6, 8, 10, None)]).to_string(index=False))print("\n[거래비용 · 가중 · 주기]")print(grid([("왕복 40bp", dict(RT_BP=40)),            ("왕복 50bp (기준)", dict()),            ("왕복 60bp", dict(RT_BP=60)),            ("역변동성 (기준)", dict()),            ("동일가중", dict(SCHEME="eq")),            ("시가총액 가중", dict(SCHEME="mc")),            ("시가총액 가중 상한 15%", dict(SCHEME="mc", WEIGHT_CAP=0.15)),            ("월간 리밸런싱", dict(FREQ="M"))]).to_string(index=False))

### 롱 신호 분해롱 사이드가 배당과 소각을 합산해야 하는지, 아니면 한쪽만으로 충분한지를 직접 확인한다.건전성 필터가 배당 지급·비감소 두 조건만일 때와, 여기에 3년 비희석까지 더했을 때를 나눠 본다.

### 코드 설명 — 신호 및 배당 필터 분해.


In [ ]:
rows = []for use_nodil in (False, True):    pool = POOL_L & (NODIL3.fillna(False) if use_nodil else True)    tag = "3개 (비희석 추가)" if use_nodil else "2개 (기준)"    for sig in ("SY", "DY", "NSI"):        res = backtest(long_signal=sig, pool_l=pool)        m, f = res["metrics"], res["ff3"]        rows.append([{"SY": "SY = DY + (−NSI)", "DY": "DY 단독", "NSI": "−NSI 단독"}[sig], tag,                     f"{m['CAGR']:.2%}", f"{m['Sharpe']:.3f}", f"{m['MDD']:.2%}",                     f"{f['coef'][0]:+.2%}", f"{f['t'][0]:.2f}"])print(pd.DataFrame(rows, columns=["롱 신호", "롱 필터", "CAGR", "샤프", "MDD",                                  "FF3 α", "t"]).to_string(index=False))

### 공매도 금지 구간 처리A는 금지월에 숏을 걷고 롱만 운용한다(채택). B는 금지를 무시하고 전 기간 롱숏을 유지한 경우로 실제 집행은불가능하지만 신호 자체의 성과를 보기 위한 참조다. C는 금지월에 KOSPI200 지수선물 매도로 숏을 대체한다.

### 코드 설명 — 공매도 금지 처리 대안 비교.


In [ ]:
mask = BANNED.reindex(DATES[:-1]).valuesrows = []for mode, lab in [("A", "A. 금지월 롱온리 (채택)"), ("B", "B. 금지 무시"), ("C", "C. 지수선물 헤지")]:    res = backtest(BAN_MODE=mode)    m, f = res["metrics"], res["ff3"]    rb, rn = res["r"][mask], res["r"][~mask]    cum = (1 + rb).prod() - 1    pn = performance_metrics(rn, rf)    rows.append([lab, f"{m['CAGR']:.2%}", f"{m['Sharpe']:.3f}", f"{m['MDD']:.2%}",                 f"{f['coef'][0]:+.2%} (t={f['t'][0]:.2f})",                 f"{cum:+.1%}", f"{pn['CAGR']:.2%} / {pn['Sharpe']:.3f}"])k_ban  = kospi[pd.Series(mask, index=RDATES).reindex(kospi.index).fillna(False)]k_free = kospi[~pd.Series(mask, index=RDATES).reindex(kospi.index).fillna(False)]pk = performance_metrics(k_free, rf)rows.append(["KOSPI", f"{mk['CAGR']:.2%}", f"{mk['Sharpe']:.3f}", f"{mk['MDD']:.2%}", "—",             f"{(1+k_ban).prod()-1:+.1%}", f"{pk['CAGR']:.2%} / {pk['Sharpe']:.3f}"])print(pd.DataFrame(rows, columns=["처리", "CAGR", "샤프", "MDD", "FF3 α",                                  "금지 31개월 누적", "금지 제외 CAGR / 샤프"]).to_string(index=False))

## 10. 수용한도종목별 주문금액을 20일 평균 거래대금의 10퍼센트 이하로 제한하고 5영업일에 걸쳐 체결한다고 가정했을 때롱북이 감당할 수 있는 규모다. 거래대금 필터를 넣는 진짜 이유가 여기 있다.

### 코드 설명 — 거래대금 기반 수용한도: 문서의 ADV20 설명과 실제 rolling 방식은 별도 점검 대상.


In [ ]:
def capacity(pool_l, days=5, adv_pct=0.10, scheme=None):    sig = SY.where(pool_l)    caps = []    for d in DATES[:-1]:        if not (d.month in (3, 6, 9, 12) or d == DATES[0]):            continue        L = select(sig, d, CFG["N_LONG"], CFG["FIN_CAP"])        w = make_weights(L, d, scheme)        a = ADV.loc[d, L]        m = a.notna()        if m.sum() == 0:            continue        caps.append((a[m] * adv_pct * days / w[m]).min())    return np.array(caps)rows = []for lab, use in [("거래대금 필터 없음", False), ("거래대금 하위 10% 배제", True)]:    if use:        thr = ADV.where(UNIV).quantile(CFG["ADV_CUT"], axis=1)        liq = ADV.ge(thr, axis=0) | ADV.isna()    else:        liq = pd.DataFrame(True, index=IDXM, columns=TICKERS)    pool = UNIV & liq & LONG_OK & DY.notna() & NSI.notna()    res = backtest(pool_l=pool, pool_s=(ALIVE & k2 & liq & NSI.notna()))    c = capacity(pool)    rows.append([lab, f"{res['metrics']['Sharpe']:.3f}", f"{res['ff3']['t'][0]:.2f}",                 f"{np.median(c)/1e8:,.0f}억", f"{c.min()/1e8:,.0f}억",                 f"{np.percentile(c,75)/1e8:,.0f}억"])print(pd.DataFrame(rows, columns=["구성", "샤프", "FF3 α t",                                  "수용한도 중앙값", "최소", "상위 25%"]).to_string(index=False))

## 11. 이중정렬 5×5두 신호가 각각 독립적으로 작동하는지를 확인한다. 매 분기 말 유니버스 종목을 배당수익률과 순주식발행각 축에서 5분위로 나눠 25개 셀을 만들고, 각 셀을 동일가중으로 보유한 결과다. 필터와 비용은 적용하지 않았다.

### 코드 설명 — 배당수익률과 순주식발행 5×5 이중정렬.


In [ ]:
cells_ret = pd.DataFrame(index=RDATES, columns=pd.MultiIndex.from_product([range(1, 6), range(1, 6)]),                         dtype=float)hold = {}for i, d in enumerate(DATES[:-1]):    if (i == 0) or d.month in (3, 6, 9, 12):        dd_ = DY.loc[d].where(UNIV.loc[d]).dropna()        nn_ = NSI.loc[d].where(UNIV.loc[d]).dropna()        com = dd_.index.intersection(nn_.index)        dq = pd.qcut(dd_[com].rank(method="first"), 5, labels=False) + 1        nq = pd.qcut(nn_[com].rank(method="first"), 5, labels=False) + 1        hold = {(a, b_): list(com[(dq == a) & (nq == b_)]) for a in range(1, 6) for b_ in range(1, 6)}    nxt = DATES[i + 1]    for k, v in hold.items():        if v:            cells_ret.loc[nxt, k] = ret.loc[nxt, v].mean()ann = lambda s: ((1 + s.dropna()).prod() ** (12 / len(s.dropna())) - 1) * 100shp = lambda s: ((s.dropna() - rf.reindex(s.dropna().index)).mean()                 / (s.dropna() - rf.reindex(s.dropna().index)).std(ddof=1) * np.sqrt(12))for title, fn, fmt in [("연평균 수익률 (%)", ann, "{:6.2f}"), ("샤프지수", shp, "{:6.2f}")]:    tab = pd.DataFrame(index=[f"DY {a}" for a in range(1, 6)],                       columns=[f"NSI {b_}" for b_ in range(1, 6)], dtype=float)    for a in range(1, 6):        for b_ in range(1, 6):            tab.iloc[a - 1, b_ - 1] = fn(cells_ret[(a, b_)])    tab["행 전체"] = [fn(pd.concat([cells_ret[(a, b_)] for b_ in range(1, 6)], axis=1).mean(axis=1))                    for a in range(1, 6)]    col = [fn(pd.concat([cells_ret[(a, b_)] for a in range(1, 6)], axis=1).mean(axis=1)) for b_ in range(1, 6)]    tab.loc["열 전체"] = col + [np.nan]    print(f"\n{title}   (행 1=저배당, 열 1=소각)")    print(tab.to_string(float_format=lambda v: fmt.format(v)))

## 12. 최근 보유 종목가장 마지막 리밸런싱 시점의 실제 바스켓이다.

### 코드 설명 — 최근 보유 종목 확인.


In [ ]:
last_d = max(base["holdings"])L, S = base["holdings"][last_d]wl = make_weights(L, last_d)print(f"{last_d.date()} 재구성\n")print("롱 (SY 상위)")for n in wl.sort_values(ascending=False).index:    print(f"  {n:<16} 비중 {wl[n]:5.2%}   DY {DY.loc[last_d,n]:6.2%}   NSI {NSI.loc[last_d,n]:+7.2%}")if S:    ws = make_weights(S, last_d) * CFG["SHORT_W"]    print("\n숏 (NSI 상위)")    for n in ws.sort_values(ascending=False).index:        print(f"  {n:<16} 비중 {-ws[n]:5.2%}   NSI {NSI.loc[last_d,n]:+7.2%}")else:    print("\n숏 없음 (공매도 금지 구간)")